# Trimorious 2.0 | M1.4.2 — Recover and scientifically screen the existing corpus

**Notebook 01 of 07 · CPU runtime**

Verify the exact 5,000-record archive and teacher database. Independently recompute supported numeric questions, cap repeated templates and preserve the original review queue. No teacher model is loaded. This is the controlled initial SFT corpus (up to 288 train + 36 validation with default caps), not a claim of 30,000 diverse examples. Teacher text, unvalidated tool traces and narrative data remain excluded. If an existing review queue is present, retain its configuration and decisions.

**After running:** open `release_v2/review_queue.csv`. Review prompt, answer, reasoning, assumptions and units; enter `approve` or `reject`, reviewer and notes. Blank rows are excluded. Do not change original text. Then open notebook 02. If the source ZIP is missing, restore it to the configured Drive folder. Do not run old teacher inference.

All stages use the same `MyDrive/trimorious_PRIVATE_m1_4_1/release_v2/` workspace. Save this notebook in Drive. Edit only the configuration and select Runtime → Run all. Never run another GPU notebook concurrently. Each notebook is self-contained and contains no credentials. GPU execution is not locally tested.

In [ ]:
# @title Configuration — RECOVER
STAGE = 'RECOVER'
USE_DRIVE = True # @param {type:'boolean'}
INSTALL_GPU_DEPS = True # @param {type:'boolean'}
PROJECT_ROOT = '/content/drive/MyDrive/trimorious_PRIVATE_m1_4_1'
SOURCE_NAME = 'trimorious_m1_4_1_PRIVATE_GPU_partial_20261127T061605946898Z.zip'
D07_FILENAME = 'trimorious_m1_3_2_PRIVATE_full_1c3d70c27cc3c9f8.zip'
# Save this benchmark ZIP in PROJECT_ROOT (or upload to /content once). Required for EVALUATE.
FRESH_TEST_PATH = '' # @param {type:'string'}
# Fresh test JSONL format documented in section 5. Must be externally authored, sealed before evaluation; preferably before training.
FRESH_TEST_SHA256 = '' # @param {type:'string'}
FRESH_INDEPENDENCE_ATTESTED = False # @param {type:'boolean'}
# Confirm external authorship, expert key review, no access to this adapter's evaluation outputs, no tuning on the test.
DATA_LICENSE_REVIEWED = False # @param {type:'boolean'}
DATA_REVIEW_APPROVED = False # @param {type:'boolean'}
# By approving, you confirm rights to train and distribute the resulting adapter, and actual review of CSV rows.
HUMAN_EVAL_APPROVED = False # @param {type:'boolean'}
# After actual rubric review; see evaluation report. Not an automatic pass.
PUBLISH_APPROVAL = '' # @param {type:'string'}
MAKE_PUBLIC = False # @param {type:'boolean'}
PUBLIC_APPROVAL = '' # @param {type:'string'}
MAX_TRAIN_PER_TEMPLATE = 12
MAX_VAL_PER_TEMPLATE = 6
MAX_LENGTH = 512
GRAD_ACCUM = 16
EPOCHS = 1
SEED = 417031
# Provisional minimum release policy: approve before training; cannot change after dataset freeze.
MIN_D07_GAIN = 1.0       # percentage points, measurable diagnostic gain
MAX_FAMILY_DROP = 5.0   # percentage points
MIN_FRESH_GAIN = 0.0
MAX_PROTOCOL_DROP = 2.0
BASE = 'Qwen/Qwen3-4B-Instruct-2507'
REV = 'cdbee75f17c01a7cc42f958dc650907174af0554'
TARGET = 'roime81/trimorious-research-4B-v2'
EXPECTED_CORPUS = '9fe5e9dbce17096c4d1ddd28a82e45111a37cbba66c574ec63473f6f70dd283a'

In [ ]:
# @title Mount Drive and prepare local worker scripts
from pathlib import Path
import json, os, sys, subprocess, hashlib, csv, sqlite3, shutil, traceback
from collections import Counter
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT = Path(PROJECT_ROOT)
assert ROOT.is_dir(), f'Project folder missing: {ROOT}'
WORK = ROOT/'release_v2'; WORK.mkdir(exist_ok=True)
SCRIPT_DIR = Path('/tmp/trimorious_v2_sequence'); SCRIPT_DIR.mkdir(exist_ok=True)
print('Persistent workspace:', WORK)
print('Local script directory:', SCRIPT_DIR)
CONFIG_NAMES=['USE_DRIVE', 'INSTALL_GPU_DEPS', 'PROJECT_ROOT', 'SOURCE_NAME', 'D07_FILENAME', 'FRESH_TEST_PATH', 'FRESH_TEST_SHA256', 'FRESH_INDEPENDENCE_ATTESTED', 'DATA_LICENSE_REVIEWED', 'DATA_REVIEW_APPROVED', 'HUMAN_EVAL_APPROVED', 'PUBLISH_APPROVAL', 'MAKE_PUBLIC', 'PUBLIC_APPROVAL', 'MAX_TRAIN_PER_TEMPLATE', 'MAX_VAL_PER_TEMPLATE', 'MAX_LENGTH', 'GRAD_ACCUM', 'EPOCHS', 'SEED', 'MIN_D07_GAIN', 'MAX_FAMILY_DROP', 'MIN_FRESH_GAIN', 'MAX_PROTOCOL_DROP', 'BASE', 'REV', 'TARGET', 'EXPECTED_CORPUS']
CONFIG={k:globals()[k] for k in CONFIG_NAMES}
assert TARGET=='roime81/trimorious-research-4B-v2' and BASE=='Qwen/Qwen3-4B-Instruct-2507' and REV=='cdbee75f17c01a7cc42f958dc650907174af0554'

## Embedded implementation
The following cells write the implementation to the local runtime. Run them unchanged. No source code download required.

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/common.py
# @title 1. Mount Drive and common helpers (CPU)
import os, sys, json, hashlib, math, re, random, csv, zipfile, sqlite3, statistics, time, gc, shutil, platform
from pathlib import Path
from collections import Counter, defaultdict
from fractions import Fraction
from importlib import metadata
assert STAGE in {'RECOVER','APPROVE','SMOKE','TRAIN','EVALUATE','PACKAGE','PUBLISH'}
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT=Path(PROJECT_ROOT); assert ROOT.is_dir(), f'Missing project folder: {ROOT}'
WORK=ROOT/'release_v2'; WORK.mkdir(exist_ok=True)
def canon(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False)
def sha(b): return hashlib.sha256(b if isinstance(b,bytes) else canon(b).encode()).hexdigest()
def put(p,obj):
    p=Path(p); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False)); tmp.replace(p)
def get(p): return json.loads(Path(p).read_text())
def lines(b): return [json.loads(x) for x in b.splitlines() if x.strip()]
def readl(p): return lines(Path(p).read_text())
def writel(p,rows):
    p=Path(p); tmp=p.with_suffix(p.suffix+'.tmp'); tmp.write_text(''.join(canon(r)+'\n' for r in rows)); tmp.replace(p)
def writecsv(p,rows,fields):
    with Path(p).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
def archive_check(z):
    ns=z.namelist(); assert len(ns)==len(set(ns)) and z.testzip() is None
    m=json.loads(z.read('archive_manifest.json')); assert set(m)==set(ns)-{'archive_manifest.json'}
    assert all(sha(z.read(k))==v for k,v in m.items()), 'Archive hash mismatch'
    return m
def state(stage,**kw):
    p=WORK/'project_state.json'; s=get(p) if p.exists() else {}; s[stage]=kw; put(p,s)
def norm(s): return re.sub(r'\s+',' ',s.casefold()).strip()
def mask(s): return re.sub(r'(?<![a-z])[-+]?\d+(?:\.\d+)?(?:e[-+]?\d+)?','NUM',norm(s))
def close(a,b): return math.isclose(float(a),float(b),rel_tol=1e-8,abs_tol=1e-10)
POLICY={'min_d07_gain_pp':MIN_D07_GAIN,'max_family_drop_pp':MAX_FAMILY_DROP,'min_fresh_gain_pp':MIN_FRESH_GAIN,'max_protocol_drop_pp':MAX_PROTOCOL_DROP}
print('Stage:',STAGE,'| Persistent work directory:',WORK)

In [ ]:
%%writefile /tmp/trimorious_v2_sequence/recover.py
def recompute(r):
    # Only supported numerical templates; never execute prompt text as code.
    family=r['family']; t=int(r['template_id'].rsplit('_',1)[1]); q=r['prompt']
    nums=[float(x) for x in re.findall(r'(?<![\d.])[-+]?\d+(?:\.\d+)?',q)]
    if family=='mathematical_reasoning':
        if t==0: a,s=nums[-2:]; return (s+math.sqrt(s*s-4*a))/2
        if t==1: _,a,_,b,_,c=nums; return b+c+a/2
        if t==2: n,k=map(int,nums); return math.comb(n-2,k-2)
        if t==3: a,b,zero,c=nums; return max(abs(a),abs(c))
        if t==4:
            _,_,a,b,c=nums; b=abs(b); x=min(1,b/(2*a)); return a*x*x-b*x+c
        if t==5: a,b,_,_,c=nums; return abs(3*a+4*b-c)/5
        if t==6: a,b,c=map(int,nums); return pow(a,b,c)
        if t==7: a,b,c,_=nums; return a*a+2*a*b*c
        if t==8: raise ValueError('Use bounded matrix parser')  # dispatched below
        if t==9: a,_,b,c=nums; return math.fsum(a/b**i for i in range(int(c)))
    if family=='scientific_reasoning':
        if t==0: a,b,c=nums; return a*b*b/(2*c)
        if t==1: a,b,c=nums; return b*c/a
        if t==2: a,b,c=nums; return b*math.exp(-math.log(2)*c/a)
        if t==3: a,b=nums; return b*a*a/10000
        if t==4: a,b,c=nums; return b*(a-c)/a
        if t==5: a,b,c=nums; return a*(c*c-b*b)/2
        if t==6: a,b,c=nums; return a*c/(b+c)
        if t==7: a,b,c=nums; return a*1e-6/(b*c)
        if t==8: a,b,c=nums; return b*math.exp(a*c)
        if t==9: a,b,c=nums; return b*c/a
    if family=='statistical_reasoning':
        if t==0: n,k,_,z=nums; return (k+z*z/2)/(n+z*z)
        if t==1: n,a,m,b=nums; return math.sqrt(a*a/n+b*b/m)
        if t==2: n,k=nums; return 2*k*(n-k)/(n*(n-1))
        if t==3: return statistics.pvariance(nums)
        if t==4: return 1/nums[0]
        if t==5: n,p=nums; return n*p*(1-p)
        if t==6: a,b=nums; return a-b
        if t==7: a,b=nums[:5],nums[5:]; return statistics.mean(abs(x-y) for x,y in zip(a,b))
        if t==8: s=sum(nums); return -sum(x/s*math.log(x/s) for x in nums)
        if t==9: a,b,c,d=nums; return (a/c+b/d)/(1/c+1/d)
    raise ValueError('Unsupported template')
# Matrix prompts contain vertex numbers and nine entries; use an explicit bounded parser.
_recompute_basic=recompute
def recompute(r):
    if r['template_id']=='mathematical_reasoning_08':
        s=r['prompt'].split('matrix ',1)[1].split(', interpreting',1)[0]
        import ast
        a=ast.literal_eval(s); assert len(a)==3 and all(len(row)==3 for row in a)
        assert all(isinstance(v,int) and 0<=v<=100 for row in a for v in row)
        return sum(a[0][i]*a[i][2] for i in range(3))
    return _recompute_basic(r)

if STAGE=='RECOVER':
    source=ROOT/SOURCE_NAME; assert source.is_file(), f'Missing {source}'
    with zipfile.ZipFile(source) as z:
        manifest=archive_check(z); summary=json.loads(z.read('reports/summary.json'))
        all_candidates=[]
        for name in z.namelist():
            if (name.startswith('candidate_datasets/') and name.endswith('.jsonl')) or name=='quarantine/records.jsonl':
                all_candidates.extend(lines(z.read(name)))
        teacher_audit=lines(z.read('teacher_review/unapproved_proposals.jsonl'))+lines(z.read('quarantine/teacher_proposals.jsonl'))
    all_candidates.sort(key=lambda r:r['id'])
    assert len(all_candidates)==len({r['id'] for r in all_candidates})==5000
    excluded={'record_sha256','d07_screen','prior_pilot_char_similarity','disposition','diversity','split'}
    assert all(sha({k:v for k,v in r.items() if k not in excluded})==r['record_sha256'] for r in all_candidates)
    assert sha([r['record_sha256'] for r in all_candidates])==summary['corpus_sha256']==EXPECTED_CORPUS
    assert summary['d07']['questions_sha256']=='e724980d46b644f47952d3ab87789e94a32cb0b79835c4721ad3c7616c311c7e'
    assert all(r['training_approved'] is False for r in all_candidates)
    assert dict(Counter(r['split'] for r in all_candidates))=={'development_candidate':4000,'validation_candidate':1000}
    assert not any(r['d07_screen']['flagged'] for r in all_candidates)
    assert {r['template_id'] for r in all_candidates if r['split']=='development_candidate'}.isdisjoint({r['template_id'] for r in all_candidates if r['split']=='validation_candidate'})
    db=ROOT/'teacher_checkpoints.sqlite3'; db_counts={}
    if db.exists():
        with sqlite3.connect(db.resolve().as_uri()+'?mode=ro',uri=True) as conn:
            assert conn.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
            db_counts=dict(conn.execute("SELECT json_extract(payload,'$.teacher'),COUNT(*) FROM results GROUP BY 1"))
    eligible=defaultdict(list); ledger=[]; seen=set()
    for r in all_candidates:
        reason='excluded_narrative_or_tool'; check=None
        if r['family'] in {'mathematical_reasoning','scientific_reasoning','statistical_reasoning'} and r['split'] in {'development_candidate','validation_candidate'}:
            try:
                independent=recompute(r); m=re.search(r'Final result: ([^ ]+) \(',r['reference_response']); assert m
                check=bool(close(independent,r['numeric_answer']) and close(independent,r['independent_reference']) and close(independent,float(m[1])))
            except Exception as e: check=False
            if not check: reason='failed_independent_recheck'
            elif r['d07_screen']['flagged'] or r['verification']['passed'] is not True: reason='failed_prior_gate'
            elif norm(r['prompt']) in seen: reason='duplicate_prompt'
            else:
                reason='eligible_for_review'; seen.add(norm(r['prompt'])); eligible[(r['split'],r['template_id'])].append(r)
        ledger.append({'id':r['id'],'template':r['template_id'],'recomputed':check,'disposition':reason})
    chosen=[]
    for (split,template), group in sorted(eligible.items()):
        random.Random(SEED+sum(map(ord,template))).shuffle(group)
        cap=MAX_TRAIN_PER_TEMPLATE if split=='development_candidate' else MAX_VAL_PER_TEMPLATE
        chosen.extend(group[:cap])
    train_groups={r['template_id'] for r in chosen if r['split']=='development_candidate'}
    val_groups={r['template_id'] for r in chosen if r['split']=='validation_candidate'}
    assert train_groups and val_groups and train_groups.isdisjoint(val_groups)
    assert len(chosen)==len({r['id'] for r in chosen})
    chosen.sort(key=lambda r:r['id'])
    queue=[dict(id=r['id'],template=r['template_id'],split=r['split'],prompt=r['prompt'],response=r['reference_response'],unit=r['unit'],decision='',reviewer='',notes='') for r in chosen]
    fingerprint=sha(chosen)
    review_path=WORK/'review_queue.csv'
    if review_path.exists():
        assert (WORK/'recovery.json').exists() and get(WORK/'recovery.json')['review_set_sha256']==fingerprint, 'Existing review differs; preserve it and use a new workspace.'
        print('Existing review CSV preserved')
    else: writecsv(review_path,queue,list(queue[0]))
    writel(WORK/'review_candidates.jsonl',chosen); writel(WORK/'recovery_ledger.jsonl',ledger); writel(WORK/'teacher_audit_private.jsonl',teacher_audit)
    report={'source':source.name,'archive_sha256':sha(source.read_bytes()),'corpus_sha256':EXPECTED_CORPUS,'record_count':len(all_candidates),'manifest_files':len(manifest),'teacher_export_counts':dict(Counter(p['teacher'] for p in teacher_audit)),'teacher_database_counts':db_counts,'teacher_in_training':0,'recomputed_counts':dict(Counter(str(x['recomputed']) for x in ledger)),'dispositions':dict(Counter(x['disposition'] for x in ledger)),'review_counts':dict(Counter(r['split'] for r in chosen)),'review_set_sha256':fingerprint,'train_templates':sorted(train_groups),'validation_templates':sorted(val_groups),'training_approved':False}
    put(WORK/'recovery.json',report); state('RECOVER',complete=True,report='recovery.json'); print(json.dumps(report,indent=2)); print('NEXT: review',review_path,'then APPROVE. Do not approve unchecked rows.')

## Execute stage
Read the actual outputs. A stopped or failed gate is not a completed stage.

In [ ]:
# @title Execute this notebook stage
def file_sha(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def read_json(p): return json.loads(Path(p).read_text())
def save_json(p,x):
    p=Path(p); temp=p.with_suffix(p.suffix+'.tmp'); temp.write_text(json.dumps(x,indent=2)); temp.replace(p)
def record_controller(stage,status,detail):
    save_json(WORK/'auto_controller_state.json',{'stage':stage,'status':status,'detail':str(detail)})
def pause(stage,detail):
    record_controller(stage,'waiting',detail); print('\nPAUSED — '+stage+'\n'+str(detail)+'\nSave changes, then Runtime → Run all. Existing artifacts are preserved.',flush=True)
def cpu(stage, parts):
    ns={**CONFIG,'USE_DRIVE':False,'STAGE':stage,'__name__':'__main__'}
    for part in ['common']+parts:
        p=SCRIPT_DIR/(part+'.py'); exec(compile(p.read_text(),str(p),'exec'),ns)
    return ns

def gpu(stage):
    print('\n=== '+stage+' — isolated GPU worker ===',flush=True)
    record_controller(stage,'running','GPU subprocess')
    config_path=SCRIPT_DIR/'run_config.json'; save_json(config_path,CONFIG)
    env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'; env.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
    p=subprocess.Popen([sys.executable,'-u',str(SCRIPT_DIR/'worker.py'),str(config_path),stage],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout:
            print(line,end='',flush=True)
        status=p.wait()
    except BaseException:
        p.terminate()
        try: p.wait(timeout=20)
        except subprocess.TimeoutExpired: p.kill(); p.wait()
        raise
    if status: raise RuntimeError(f'{stage} worker exited with code {status}. Read the actual error above; checkpoints preserved.')
    # Child exit releases CUDA allocations. Dependencies already installed for subsequent child processes in this session.
    CONFIG['INSTALL_GPU_DEPS']=False

def validate_active(ns):
    approval,run,done,final=ns['active_run']()
    cfg=done['config']
    for k,v in {'base':BASE,'revision':REV,'max_length':MAX_LENGTH,'grad_accum':GRAD_ACCUM,'epochs':EPOCHS,'seed':SEED}.items():
        assert cfg[k]==v,f'Trained adapter uses different {k}; do not silently change a finished run.'
    assert done['steps']>0 and (final/'adapter_model.safetensors').is_file()
    return approval,run,done,final

def get_eval_artifacts(run):
    active=read_json(WORK/'active_evaluation.json'); assert active['run']==run.name
    folder=run/'evaluation'/active['evaluation']; return folder,read_json(folder/'evaluation_report.json')

def eval_receipt(run,done):
    folder,report=get_eval_artifacts(run)
    assert report['evaluation_config']['adapter_hashes']==done['adapter_hashes']
    assert report['evaluation_config']['fresh_hash']==FRESH_TEST_SHA256 and report['fresh']
    db=folder/'responses.sqlite3'
    with sqlite3.connect(db.resolve().as_uri()+'?mode=ro',uri=True) as conn:
        assert conn.execute('PRAGMA integrity_check').fetchone()[0]=='ok'
        counts=dict(conn.execute('SELECT model,COUNT(*) FROM responses GROUP BY model'))
        expected=1000+report['fresh']['n']
        assert counts=={'qwen3_base':expected,'trimorious_v2':expected}, f'Incomplete paired evaluation: {counts}'
    return {'evaluation':folder.name,'report_sha256':file_sha(folder/'evaluation_report.json'),'db_sha256':file_sha(db),'mapping_sha256':file_sha(folder/'rubric_mapping_private.json'),'fresh_sha256':FRESH_TEST_SHA256}



def audit_stage(stage, paths):
    from datetime import datetime, timezone
    target=WORK/'stage_audits'; target.mkdir(exist_ok=True)
    files=[Path(p) for p in paths if Path(p).is_file()]
    for p in files: assert p.resolve().is_relative_to(WORK.resolve())
    manifest={str(p.relative_to(WORK)):file_sha(p) for p in files}
    stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    dest=target/f'{stage}_{stamp}_PRIVATE.zip'; temp=dest.with_suffix('.tmp')
    import zipfile
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as z:
        for p in files: z.write(p,str(p.relative_to(WORK)))
        z.writestr('audit_manifest.json',json.dumps(manifest,indent=2))
    with zipfile.ZipFile(temp) as z:
        assert z.testzip() is None
        assert all(hashlib.sha256(z.read(k)).hexdigest()==v for k,v in manifest.items())
    temp.replace(dest); print('Private stage audit:',dest)

ns=cpu('RECOVER',['recover'])
audit_stage(STAGE,[WORK/'recovery.json',WORK/'project_state.json'])
print('NEXT: review the CSV, then notebook 02.')